[Back to Computer Networks guideline](Computer-Networks.html)


## **Reliable End-to-End Transport**

The previous chapters followed packets across links, routers, forwarding tables, and routing protocols. Reaching the correct host is still not enough. A laptop can run a browser, a DNS resolver, a video call, and a software update at the same time. The transport layer must deliver each arriving payload to the correct **process**, and it must provide the communication behavior that the application expects.

This chapter continues the HTTPS example. A DNS query may use UDP because a small request and response fit a message-oriented exchange. The HTTPS connection then needs ordered, reliable application bytes. It can obtain them from TCP, or HTTP/3 can obtain independent reliable streams from QUIC over UDP. These protocols face the same underlying uncertainty: packets can be corrupted, lost, duplicated, delayed, or reordered.

::: {.callout-note}
On a first reading, focus on four ideas: a socket identifies an endpoint, sequence numbers identify data, acknowledgments report receiver progress, and a window permits several packets to be in flight. Then connect TCP's byte sequence numbers, retransmission timer, receive window, and connection state machine. QUIC is easier to understand after that foundation.
:::

Reliable transport is not "send every packet twice." It is a state machine that distinguishes new data from duplicates, infers loss without seeing the path directly, retransmits only when necessary, and prevents a fast sender from overflowing a slow receiver. Each mechanism solves one ambiguity, and the mechanisms must remain correct when their own control packets are lost.


### **Transport-Layer Services**

#### **Process-to-Process Communication**

IP provides **host-to-host** delivery: an IP address identifies an interface or network endpoint. Transport protocols extend this to **process-to-process** delivery. TCP and UDP place 16-bit source and destination **port numbers** around the application data. The destination port tells the receiving operating system which socket should receive the payload; the source port supplies a return endpoint and helps distinguish concurrent conversations.

A useful analogy is an apartment building. The IP address is the street address, while the port is an apartment or service desk inside the building. The analogy has limits: a port is an operating-system namespace, not a permanently assigned physical room, and one server port can support many simultaneous TCP connections.

#### **Ports, Sockets, Multiplexing, and Demultiplexing**

A **socket** is the operating system's application-facing communication object. It contains protocol state and is bound or connected to endpoint information. A port number is only one field of that state, so "socket" and "port" are not synonyms.

For an established Internet flow, the common classifier is the **five-tuple**

$$
(\text{transport protocol},\ \text{source IP},\ \text{source port},\ \text{destination IP},\ \text{destination port}).
$$

The protocol is necessary because TCP port 53 and UDP port 53 occupy different namespaces. For TCP, a server first has a listening socket such as `192.0.2.80:443`. Each successful connection creates an accepted socket with a distinct remote address and port, so thousands of clients can share the same local server port. UDP commonly demultiplexes primarily by local endpoint, although a connected UDP socket can also restrict the accepted peer. Calling `connect()` on UDP does **not** perform a network handshake; it records a default peer and local filtering state.

**Multiplexing** combines data from many application sockets onto the network. **Demultiplexing** reverses the operation when packets arrive.

![Transport demultiplexing delivers simultaneous flows to the correct local socket and process.](assets/transport-demultiplexing.svg){fig-alt="UDP and TCP flows are classified by protocol, addresses, and ports and delivered to DNS, web, and API processes" width="98%"}

The diagram also explains why a packet capture should be filtered by a flow tuple rather than only by destination port. Two clients can contact the same web server port while retaining independent sequence numbers, windows, timers, and connection states.

#### **Connectionless and Connection-Oriented Services**

A **connectionless** transport lets an application send a message without first establishing shared transport state. UDP follows this model. Each datagram carries enough addressing information to be handled independently. This reduces protocol setup, but UDP itself does not remember which messages arrived or repair losses.

A **connection-oriented** transport establishes endpoint state before normal data transfer. TCP's connection is not a reserved physical circuit through the Internet. It is synchronized state at the two endpoints: sequence spaces, acknowledgment progress, windows, options, and timers. Routers can forward TCP packets without knowing that this end-to-end connection exists.

#### **Transport Guarantees and Application Requirements**

Choosing a transport starts with application semantics, not with a slogan that one protocol is "faster."

| Application need | Important question | Typical consequence |
|---|---|---|
| Message boundaries | Must one send remain one receiveable unit? | UDP preserves datagrams; TCP requires application framing |
| Reliability | Can missing data be ignored, replaced, or must it arrive? | File transfer needs repair; live voice may prefer timely newer audio |
| Ordering | Must later data wait for an earlier gap? | TCP orders one byte stream; QUIC orders independently within each stream |
| Latency | Is stale data worse than missing data? | Interactive media may avoid retransmitting expired samples |
| Security | Are authentication and confidentiality required? | TLS over TCP or QUIC's integrated TLS protects application traffic |
| Rate adaptation | Can the sender overwhelm the receiver or path? | Flow control and congestion control are necessary |

DNS, real-time media, games, telemetry, file transfer, and web browsing therefore make different tradeoffs. UDP is suitable only when the application or a protocol above it supplies every additional behavior it needs. [RFC 8085](https://datatracker.ietf.org/doc/html/rfc8085) emphasizes that Internet applications using UDP still need congestion-responsive behavior; UDP is a minimal substrate, not permission to transmit without restraint.


In [1]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Flow:
    protocol: str
    source_ip: str
    source_port: int
    destination_ip: str
    destination_port: int


listeners = {
    ("UDP", "192.0.2.80", 53): "DNS socket",
    ("TCP", "192.0.2.80", 443): "HTTPS listener",
    ("TCP", "192.0.2.80", 8080): "API listener",
}

# An established TCP connection has a more specific five-tuple entry.
established = {
    Flow("TCP", "10.0.0.8", 53001, "192.0.2.80", 443): "HTTPS worker A",
    Flow("TCP", "10.0.0.9", 53002, "192.0.2.80", 443): "HTTPS worker B",
}


def demultiplex(flow: Flow) -> str:
    """Prefer an exact connection; otherwise fall back to a listener."""
    if flow in established:
        return established[flow]
    listener_key = (flow.protocol, flow.destination_ip, flow.destination_port)
    return listeners.get(listener_key, "no matching socket")


arrivals = [
    Flow("UDP", "10.0.0.8", 53000, "192.0.2.80", 53),
    Flow("TCP", "10.0.0.8", 53001, "192.0.2.80", 443),
    Flow("TCP", "10.0.0.9", 53002, "192.0.2.80", 443),
    Flow("TCP", "10.0.0.8", 53003, "192.0.2.80", 8080),
]

for arrival in arrivals:
    print(f"{arrival.protocol:3} destination :{arrival.destination_port:<4} -> {demultiplex(arrival)}")


UDP destination :53   -> DNS socket
TCP destination :443  -> HTTPS worker A
TCP destination :443  -> HTTPS worker B
TCP destination :8080 -> API listener


### **User Datagram Protocol**

UDP is a minimal, message-oriented transport over IP. [RFC 768](https://datatracker.ietf.org/doc/html/rfc768) defines only four 16-bit header fields: source port, destination port, length, and checksum. Its fixed header is eight bytes.

#### **UDP Header and Checksum**

![The UDP header contains source port, destination port, datagram length, and checksum.](assets/udp-header.svg){fig-alt="UDP header with four sixteen-bit fields followed by data" width="72%"}

*Figure source: [DnaX, Header of UDP, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Header_of_UDP.svg), licensed under CC BY-SA 4.0.*

- **Source port** identifies the sender's return endpoint when meaningful. A zero source port is possible in IPv4 UDP, although ordinary applications use an assigned ephemeral or service port.
- **Destination port** selects the receiving service.
- **Length** counts the UDP header and payload in octets, so its minimum ordinary value is 8.
- **Checksum** detects corruption over a pseudo-header, UDP header, and payload. The pseudo-header includes source and destination IP addresses, protocol number, and UDP length, so some addressing corruption is also detected.

The checksum uses one's-complement arithmetic. An odd-length input is padded for calculation only. A checksum verifies integrity; it does not repair damage, prove sender identity, or encrypt data. IPv4 historically permits a transmitted zero checksum to mean "not calculated." Ordinary IPv6 UDP requires a checksum, apart from narrowly specified exceptions.

#### **Datagram Boundaries and Message Loss**

One UDP `sendto()` creates one datagram, and a receiver obtains datagrams individually with `recvfrom()`. A zero-length datagram is therefore a real message, not an end-of-file marker. If the application's receive buffer is too small, the datagram can be truncated rather than continued by the next call.

UDP does not guarantee delivery, duplicate suppression, ordering, retransmission, receiver flow control, or path congestion control. If an application sends messages `A`, `B`, and `C`, it may observe `B`, `A`, `C`, two copies of `B`, or no message at all. The application must attach an identifier if it needs to detect any of those outcomes.

A UDP datagram is normally carried in one IP packet. Oversized messages can trigger IP fragmentation or simply fail when Path MTU Discovery is ineffective. Because losing one fragment loses the complete datagram and fragments are awkward for middleboxes, [RFC 8085](https://datatracker.ietf.org/doc/html/rfc8085#section-3.2) recommends designing message sizes to avoid fragmentation.

#### **When UDP Is the Right Abstraction**

UDP is useful when message boundaries are natural and the application wants direct control over timing or repair. Examples include small DNS transactions, real-time media, multiplayer game state, measurement probes, multicast, and protocols such as QUIC that implement richer transport behavior in user space.

It is misleading to say that UDP is automatically low-latency. A protocol over UDP can still handshake, retransmit, encrypt, pace, and control congestion. Its advantage is that these choices are not imposed by UDP's own byte-stream state machine. Its cost is that implementing them correctly is difficult.

| Property | UDP itself | Application or upper protocol may add |
|---|---|---|
| Message boundaries | Yes | Message schema and identifiers |
| Checksum | Yes | Cryptographic integrity and authentication |
| Reliable delivery | No | ACKs, timers, retransmission, or erasure coding |
| Ordered delivery | No | Sequence numbers and reassembly |
| Flow control | No | Receiver credit or backpressure |
| Congestion control | No | Rate adaptation and pacing |


In [2]:
import ipaddress
import struct


def ones_complement_sum(data: bytes) -> int:
    """Return the folded one's-complement sum used by UDP and TCP."""
    if len(data) % 2:
        data += b"\x00"  # Padding participates in the sum, not the packet length.
    total = sum(struct.unpack(f"!{len(data) // 2}H", data))
    while total >> 16:
        total = (total & 0xFFFF) + (total >> 16)
    return total


def internet_checksum(data: bytes) -> int:
    result = (~ones_complement_sum(data)) & 0xFFFF
    return 0xFFFF if result == 0 else result


def build_udp_segment(source_ip: str, destination_ip: str,
                      source_port: int, destination_port: int,
                      payload: bytes) -> bytes:
    udp_length = 8 + len(payload)
    header_without_checksum = struct.pack(
        "!HHHH", source_port, destination_port, udp_length, 0
    )
    pseudo_header = (
        ipaddress.IPv4Address(source_ip).packed
        + ipaddress.IPv4Address(destination_ip).packed
        + struct.pack("!BBH", 0, 17, udp_length)  # 17 is UDP's IP protocol number.
    )
    checksum = internet_checksum(pseudo_header + header_without_checksum + payload)
    return struct.pack(
        "!HHHH", source_port, destination_port, udp_length, checksum
    ) + payload


payload = b"transport"
segment = build_udp_segment("192.0.2.10", "198.51.100.20", 53000, 53, payload)
source_port, destination_port, udp_length, checksum = struct.unpack("!HHHH", segment[:8])

pseudo = (
    ipaddress.IPv4Address("192.0.2.10").packed
    + ipaddress.IPv4Address("198.51.100.20").packed
    + struct.pack("!BBH", 0, 17, udp_length)
)
verified = ones_complement_sum(pseudo + segment) == 0xFFFF

print(f"ports: {source_port} -> {destination_port}")
print(f"UDP length: {udp_length} bytes; payload: {len(payload)} bytes")
print(f"checksum: 0x{checksum:04x}; verification passed: {verified}")


ports: 53000 -> 53
UDP length: 17 bytes; payload: 9 bytes
checksum: 0x1778; verification passed: True


### **Principles of Reliable Data Transfer**

#### **Bit Errors, Loss, Duplication, and Reordering**

Reliability begins by naming the channel's possible failures:

- a **bit error** changes the packet's contents;
- **loss** makes a sent packet invisible to the receiver;
- **duplication** presents the same logical packet more than once;
- **reordering** makes a later packet arrive before an earlier one;
- variable delay makes a slow packet look indistinguishable from a lost packet for some period.

The last point is fundamental. A sender cannot directly inspect every router and prove that a packet is gone. It chooses a timeout or another loss signal and acts on an inference. A correct protocol must tolerate an incorrect inference, because the original and a retransmission can both arrive.

#### **Checksums, Sequence Numbers, ACKs, and NACKs**

A checksum answers "does this packet appear intact?" A **sequence number** answers "which logical data is this?" An acknowledgment answers "what has the receiver accepted?" These fields solve different problems.

A positive acknowledgment (**ACK**) can name one packet, the highest contiguous packet, the next expected byte, or explicit received ranges. A negative acknowledgment (**NACK**) explicitly reports a gap or corrupt packet. NACKs can accelerate repair but cannot be the only loss mechanism: the NACK itself can be lost, and a receiver cannot report a packet it does not yet know exists.

#### **Timers and Retransmissions**

A sender retains unacknowledged data. If suitable evidence of progress does not arrive, it retransmits. The timer must exceed normal delay often enough to avoid waste, yet remain short enough to recover from real loss. Later sections show how TCP estimates this value from RTT samples.

Retransmission creates ambiguity. If an ACK is lost, the sender does not know that the first transmission succeeded. The receiver therefore needs sequence numbers and duplicate suppression.

![In the alternating-bit protocol, a lost ACK causes retransmission, but the receiver suppresses duplicate delivery.](assets/reliable-transfer-timeline.svg){fig-alt="Sender receiver timeline where ACK zero is lost, data zero is retransmitted, and the duplicate is not delivered twice" width="94%"}

#### **Protocol State Machines and Safety Properties**

A transport protocol is most clearly specified as interacting state machines. State determines which packet is legal and how an event changes local variables.

```text
SENDER(message)
    packet <- checksum(sequence, message)
    repeat
        transmit(packet)
        wait until a valid matching ACK arrives or the timer expires
    until matching ACK arrived
    toggle sequence number

RECEIVER(packet)
    if packet is intact and packet.sequence == expected
        deliver payload exactly once
        remember ACK for this sequence
        toggle expected sequence number
    send ACK for the last correctly accepted sequence
```

Two classes of properties help evaluate the design:

- **Safety:** nothing bad happens. A byte is not corrupted, reordered at the application boundary, or delivered twice.
- **Liveness:** something good eventually happens. If the path eventually carries packets and ACKs, queued data eventually reaches the application.

A protocol can be safe but useless if it waits forever. It can be live but incorrect if it delivers duplicates. Reliability requires both under an explicit channel model.


In [3]:
def alternating_bit_with_lost_ack(message: str):
    """Trace one transfer where the first ACK is lost."""
    sender_sequence = 0
    receiver_expected = 0
    delivered = []
    events = []

    # First transmission reaches the receiver.
    events.append(f"send DATA({sender_sequence}, {message})")
    if sender_sequence == receiver_expected:
        delivered.append(message)
        receiver_expected ^= 1
        events.append("receiver delivers message and sends ACK(0)")

    # The channel loses ACK(0), so the sender cannot safely advance.
    events.append("channel loses ACK(0); sender timer expires")
    events.append(f"retransmit DATA({sender_sequence}, {message})")

    # The receiver now expects 1, so sequence 0 is a recognizable duplicate.
    if sender_sequence != receiver_expected:
        events.append("receiver suppresses duplicate and repeats ACK(0)")

    events.append("sender receives ACK(0) and advances to sequence 1")
    return events, delivered


trace, application_deliveries = alternating_bit_with_lost_ack("A")
for step, event in enumerate(trace, start=1):
    print(f"{step}. {event}")
print("application deliveries:", application_deliveries)


1. send DATA(0, A)
2. receiver delivers message and sends ACK(0)
3. channel loses ACK(0); sender timer expires
4. retransmit DATA(0, A)
5. receiver suppresses duplicate and repeats ACK(0)
6. sender receives ACK(0) and advances to sequence 1
application deliveries: ['A']


### **Stop-and-Wait Protocols**

#### **Alternating-Bit Protocol**

**Stop-and-wait** permits only one unacknowledged data packet. The sender transmits, starts a timer, and waits. The alternating-bit protocol needs only sequence values 0 and 1 because no second new packet can be outstanding. Once packet 0 is acknowledged, a later packet with 0 must belong to a new cycle only after packet 1 has completed.

This small protocol already contains the core of reliable transport: checksum, sequence number, ACK, retransmission timer, retained sender data, and duplicate suppression. It is useful on simple links and as a proof model. Its main limitation is performance rather than correctness.

#### **Utilization and the Bandwidth-Delay Product**

Let a data packet contain $L$ bits, the bottleneck rate be $R$ bits per second, and the round-trip time be $RTT$. Ignoring ACK transmission and processing time, packet transmission takes

$$
T_{tx}=\frac{L}{R},
$$

while stop-and-wait sender utilization is approximately

$$
U_{sender}=\frac{T_{tx}}{RTT+T_{tx}}.
$$

On a 100 Mb/s path with a 40 ms RTT and a 1500-byte packet, transmission takes only 0.12 ms. The sender then waits about 40 ms, so useful utilization is roughly 0.3 percent and throughput is about 0.299 Mb/s. The link is not slow; the protocol leaves it idle.

The **bandwidth-delay product** estimates how much data can occupy the path:

$$
BDP=R\times RTT.
$$

Here the BDP is 4,000,000 bits, or 500,000 bytes. Roughly 334 full 1500-byte packets must be allowed in flight to cover that round trip.

![Stop-and-wait leaves the sender idle, whereas a pipelined window keeps several packets in flight.](assets/stop-and-wait-vs-pipeline.svg){fig-alt="Two timelines compare one packet per round trip with a pipeline of several packets" width="98%"}

#### **Why One Outstanding Packet Is Not Enough**

High-rate or long-distance paths have a large BDP. Waiting for every ACK serializes transmission around feedback latency. The remedy is not to remove acknowledgments; it is to separate "may transmit more" from "oldest data is acknowledged" by allowing a bounded **window** of outstanding data.

A window should be large enough to keep the path busy, but not unbounded. It is constrained by sequence-number safety, sender memory, receiver buffering, and network congestion. The next section isolates the first two; TCP later combines receiver and network limits.


In [4]:
from math import ceil


def stop_and_wait_metrics(rate_mbps: float, rtt_ms: float, packet_bytes: int):
    rate_bps = rate_mbps * 1_000_000
    packet_bits = packet_bytes * 8
    transmission_s = packet_bits / rate_bps
    rtt_s = rtt_ms / 1_000
    utilization = transmission_s / (rtt_s + transmission_s)
    throughput_mbps = utilization * rate_mbps
    bdp_bytes = rate_bps * rtt_s / 8
    packets_for_bdp = ceil(bdp_bytes / packet_bytes)
    return transmission_s * 1_000, utilization, throughput_mbps, bdp_bytes, packets_for_bdp


scenarios = [
    (100, 40, 1500),
    (1000, 2, 1500),
    (1000, 80, 1500),
]

print("rate   RTT   transmit   utilization   throughput   BDP window")
for rate, rtt, packet_size in scenarios:
    tx_ms, utilization, throughput, bdp_bytes, window = stop_and_wait_metrics(
        rate, rtt, packet_size
    )
    print(
        f"{rate:4}M  {rtt:3}ms  {tx_ms:7.3f}ms   {utilization:9.3%}   "
        f"{throughput:8.3f}M   {bdp_bytes / 1000:7.1f}KB ({window} packets)"
    )


rate   RTT   transmit   utilization   throughput   BDP window
 100M   40ms    0.120ms      0.299%      0.299M     500.0KB (334 packets)
1000M    2ms    0.012ms      0.596%      5.964M     250.0KB (167 packets)
1000M   80ms    0.012ms      0.015%      0.150M   10000.0KB (6667 packets)


### **Pipelined Reliable Transfer**

#### **Sliding Windows**

A sliding-window sender divides sequence space into four regions: data already acknowledged, data sent but not yet acknowledged, data currently permitted to send, and data outside the window. As ACKs advance the left edge, new sequence numbers enter on the right.

![A sliding window advances through a finite sequence-number space as packets are acknowledged.](assets/sliding-window.svg){fig-alt="Sliding window sequence space showing packets inside and outside the active window" width="64%"}

*Figure source: [Alexander Krivacs Schroder, Sliding Window, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Sliding_Window.svg), licensed under CC BY-SA 3.0.*

The sender must retain every unacknowledged packet because it may need retransmission. The receiver's policy determines whether out-of-order packets are discarded or buffered. Those choices produce two classic protocols.

#### **Go-Back-N**

Go-Back-N (GBN) permits up to $N$ unacknowledged packets at the sender but gives the receiver a one-packet window. The receiver accepts only its next expected sequence number and sends a **cumulative ACK** for the highest contiguous packet. The sender commonly runs one timer for the oldest unacknowledged packet.

```text
GBN_SENDER_ON_TIMEOUT
    retransmit every packet from send_base through next_sequence - 1

GBN_RECEIVER(packet)
    if packet is intact and packet.sequence == expected
        deliver packet; expected <- expected + 1
    send ACK for expected - 1
```

![When packet 2 is lost, Go-Back-N discards later out-of-order data and retransmits the remaining window after timeout.](assets/go-back-n.svg){fig-alt="Go-Back-N timeline with packet two lost, packet three discarded, duplicate ACK one, and retransmission from packet two" width="94%"}

GBN keeps receiver state small, but one loss can retransmit data that already crossed the network. This cost becomes large when the window or BDP is large.

#### **Selective Repeat**

Selective Repeat (SR) lets the receiver accept and buffer packets inside its receive window even when an earlier packet is missing. It acknowledges packets or ranges individually. The sender retransmits only missing data and tracks more detailed per-packet state.

![Selective Repeat buffers packets that arrive after a gap and retransmits only the missing packet.](assets/selective-repeat.svg){fig-alt="Selective Repeat sliding window timeline with an isolated missing packet and targeted retransmission" width="78%"}

*Figure source: [ArielGlenn and Lady 6thofAu, Selective Repeat, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Selectiverepeat.svg), dual-licensed under CC BY-SA 3.0 and GFDL.*

```text
SR_RECEIVER(packet)
    if packet is intact and sequence is inside receive window
        buffer it if it is new; ACK that sequence
        while the next expected packet is buffered
            deliver it and slide receive window
```

SR uses bandwidth efficiently under isolated loss and reordering, but it needs buffer space, range bookkeeping, and careful sequence-number arithmetic.

#### **Sequence-Space and Window-Size Constraints**

Sequence numbers eventually wrap. If the active windows are too large, an old delayed packet can carry the same number as a new packet and become indistinguishable. With a $k$-bit sequence number field, the space has $2^k$ values.

- For GBN with receiver window 1, the common constraint is $W_s\le 2^k-1$.
- For Selective Repeat with equal sender and receiver windows $W$, the common safe constraint is $W\le 2^{k-1}$.
- More generally, avoiding overlap requires $W_s+W_r\le 2^k$ under the usual model.

These are correctness constraints, not performance suggestions. A larger window can be worse if the sequence space cannot distinguish generations.

#### **Go-Back-N vs Selective Repeat**

| Dimension | Go-Back-N | Selective Repeat |
|---|---|---|
| Receiver accepts out of order | No | Yes, within its window |
| Acknowledgment style | Cumulative | Individual packets or ranges |
| Retransmission after one gap | Gap and all later outstanding packets | Missing packet or range only |
| Receiver buffering | Minimal | Potentially one receive window |
| Sender state | Base, next sequence, outstanding packets | Detailed packet/range state and timers |
| Best fit | Simpler channels or small windows | Large BDP, reordering, or costly retransmission |

TCP historically exposes cumulative acknowledgments and can add selective acknowledgment blocks. It is not a literal textbook SR implementation, but its modern loss recovery adopts the same principle: report received ranges so the sender can repair gaps selectively.


In [5]:
def compare_recovery(total_packets: int, lost_packet: int):
    """Count transmissions for one isolated first-round loss."""
    initial = list(range(total_packets))

    # GBN receives packets after the gap but discards them. When the
    # oldest outstanding timer expires, all packets from the gap onward replay.
    gbn_retransmissions = list(range(lost_packet, total_packets))

    # SR buffers later packets and retransmits only the missing one.
    sr_retransmissions = [lost_packet]

    return {
        "GBN": initial + gbn_retransmissions,
        "Selective Repeat": initial + sr_retransmissions,
    }


traces = compare_recovery(total_packets=6, lost_packet=2)
for protocol, transmissions in traces.items():
    print(f"{protocol:16}: {transmissions} ({len(transmissions)} transmissions)")

avoided = len(traces["GBN"]) - len(traces["Selective Repeat"])
print("transmissions avoided by selective recovery:", avoided)


GBN             : [0, 1, 2, 3, 4, 5, 2, 3, 4, 5] (10 transmissions)
Selective Repeat: [0, 1, 2, 3, 4, 5, 2] (7 transmissions)
transmissions avoided by selective recovery: 3


### **Transmission Control Protocol**

TCP, consolidated in [RFC 9293](https://datatracker.ietf.org/doc/html/rfc9293), provides a reliable, in-order, full-duplex **byte stream** between two endpoints. It does not preserve application message boundaries. Two calls to `send()` may be returned by one `recv()`, and one send may require several receives. Applications therefore need framing such as a fixed size, delimiter, length prefix, or self-describing format.

#### **Byte Streams and TCP Segments**

TCP assigns a sequence number to each byte, groups consecutive bytes into segments, and may regroup them during retransmission. Segment boundaries are implementation and path decisions, not part of the application contract. The Maximum Segment Size (MSS) option describes the largest TCP payload the peer should place in one segment for the relevant direction.

Both endpoints can send at once. Each direction has its own sequence space, ACK progress, send buffer, receive buffer, and FIN. A TCP connection is therefore identified by its endpoint pair but carries two coordinated one-way byte streams.

#### **TCP Header Fields**

![The TCP header carries ports, byte sequence state, flags, receive window, checksum, and options.](assets/tcp-header.png){fig-alt="TCP header layout with source and destination ports, sequence number, acknowledgment number, flags, window, checksum, options, and data" width="78%"}

*Figure source: [Ere, TCP header, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:TCP_header.png), released into the public domain.*

Important fields are:

- **Source and destination ports** identify endpoint applications.
- **Sequence number** identifies the first byte in this segment, except for special SYN semantics.
- **Acknowledgment number** is valid when `ACK` is set and names the next byte expected contiguously.
- **Data offset** gives the TCP header length because options make it variable.
- **Flags** include `SYN`, `ACK`, `FIN`, `RST`, `PSH`, `URG`, and ECN-related bits.
- **Window** advertises receive capacity; window scaling can extend its effective range.
- **Checksum** covers a pseudo-header, TCP header, options, and data.
- **Options** can negotiate MSS, window scale, SACK permission, and timestamps.

#### **Sequence and Acknowledgment Numbers**

If a segment has `SEQ=1000` and carries 500 bytes, it covers the half-open interval $[1000,1500)$. An `ACK=1500` means every byte before 1500 has been accepted contiguously and byte 1500 is wanted next. Sequence arithmetic wraps modulo $2^{32}$; implementations compare numbers within bounded windows rather than with ordinary unbounded integer ordering.

SYN and FIN each consume one position in sequence space even without application payload. This lets their receipt and retransmission participate in the same acknowledgment machinery.

![TCP keeps its cumulative ACK at the first missing byte while SACK reports a later range that arrived out of order.](assets/tcp-sequence-ack.svg){fig-alt="TCP timeline with bytes 1500 to 1999 lost, bytes 2000 to 2499 buffered, cumulative ACK 1500, and a SACK block" width="96%"}

#### **Cumulative ACKs and Selective Acknowledgment**

A cumulative ACK is compact and robust: `ACK=X` confirms the complete prefix below $X$. It cannot by itself describe which later ranges arrived. After several losses, a sender may otherwise rediscover gaps one round trip at a time.

The Selective Acknowledgment option defined by [RFC 2018](https://datatracker.ietf.org/doc/html/rfc2018) reports non-contiguous received blocks. SACK does not replace the cumulative acknowledgment; it adds information about data beyond the first gap. This allows the sender to preserve successfully delivered ranges and focus retransmission on missing bytes.


In [6]:
class TcpLikeReceiver:
    """Small byte-range reassembler with cumulative ACK and SACK output."""

    def __init__(self, initial_sequence: int):
        self.next_expected = initial_sequence
        self.buffered = []  # Disjoint half-open intervals [start, end).

    def receive(self, sequence: int, length: int):
        self.buffered.append((sequence, sequence + length))
        self.buffered.sort()

        # Merge overlapping or adjacent intervals.
        merged = []
        for start, end in self.buffered:
            if not merged or start > merged[-1][1]:
                merged.append([start, end])
            else:
                merged[-1][1] = max(merged[-1][1], end)
        self.buffered = [tuple(interval) for interval in merged]

        # Consume any interval that starts at or before the next wanted byte.
        advanced = True
        while advanced:
            advanced = False
            remaining = []
            for start, end in self.buffered:
                if start <= self.next_expected < end:
                    self.next_expected = end
                    advanced = True
                elif end > self.next_expected:
                    remaining.append((start, end))
            self.buffered = remaining

        sacks = [interval for interval in self.buffered if interval[0] > self.next_expected]
        return self.next_expected, sacks


receiver = TcpLikeReceiver(initial_sequence=1000)
arrivals = [(1000, 500), (2000, 500), (1500, 500)]

for sequence, length in arrivals:
    ack, sack_blocks = receiver.receive(sequence, length)
    print(
        f"received [{sequence}, {sequence + length}) -> "
        f"ACK={ack}, SACK={sack_blocks or 'none'}"
    )


received [1000, 1500) -> ACK=1500, SACK=none
received [2000, 2500) -> ACK=1500, SACK=[(2000, 2500)]
received [1500, 2000) -> ACK=2500, SACK=none


### **TCP Connection Lifecycle**

#### **Three-Way Handshake**

TCP must synchronize two independently chosen initial sequence numbers and verify that both directions can carry current packets. The three-way handshake does this before normal application transfer.

![The TCP three-way handshake synchronizes both initial sequence numbers and moves client and server into ESTABLISHED state.](assets/tcp-three-way-handshake.svg){fig-alt="Client sends SYN sequence x, server returns SYN ACK sequence y acknowledgment x plus one, and client acknowledges y plus one" width="88%"}

1. The active opener sends `SYN, SEQ=x` and enters `SYN-SENT`.
2. A listening server allocates connection state, sends `SYN+ACK, SEQ=y, ACK=x+1`, and enters `SYN-RECEIVED`.
3. The client sends `ACK=y+1`; both sides can enter `ESTABLISHED`.

Two messages would not let the server know that its own SYN reached the client. The third message confirms the reverse sequence space. Options such as MSS, window scale, SACK permission, and timestamps are also negotiated on SYN segments, so a lost option cannot silently appear halfway through a connection.

#### **Connection State Machine**

A TCP implementation reacts to packets and application calls according to state. Important states include `CLOSED`, `LISTEN`, `SYN-SENT`, `SYN-RECEIVED`, `ESTABLISHED`, `FIN-WAIT-1`, `FIN-WAIT-2`, `CLOSE-WAIT`, `CLOSING`, `LAST-ACK`, and `TIME-WAIT`.

The state machine prevents a plausible flag combination from being accepted in the wrong context. For example, an ACK that would be meaningful in `ESTABLISHED` cannot create a connection from `CLOSED`. Implementations must also validate whether sequence numbers fall inside the current receive window before acting on segments.

#### **Graceful Close, Reset, and Half-Open Connections**

TCP closes each direction independently. A FIN means "I will send no bytes after this sequence position." Receiving a FIN produces end-of-file only after all earlier bytes have been delivered. The other endpoint can continue sending, creating a **half-closed** connection.

A normal full close can therefore use four logical messages: FIN, ACK, FIN, ACK. The two middle actions may be combined when the application closes promptly. An `RST` is different: it aborts state and tells the peer that the connection cannot continue. Unsafely treating RST as a graceful end can turn truncated application data into an apparently successful transfer.

A **half-open** connection exists when one endpoint believes the connection exists but the other has lost or discarded its state, perhaps after a reboot. Retransmission, keepalive where enabled, application heartbeats, or an RST in response to unexpected traffic can expose the mismatch.

#### **TIME_WAIT and Delayed Segments**

The endpoint performing the active close normally remains in `TIME-WAIT` for twice the Maximum Segment Lifetime. This serves two purposes: it can retransmit the final ACK if the peer repeats its FIN, and it prevents delayed segments from an old connection instance from contaminating a new connection with the same endpoint tuple.

`TIME-WAIT` is therefore evidence that cleanup is working, not automatically a leak. A server with many short connections may choose application patterns and connection reuse carefully, but bypassing the state machine can trade a visible resource cost for subtle correctness failures.


### **Loss Detection and Retransmission**

#### **RTT Estimation and Retransmission Timeout**

A fixed retransmission timeout performs poorly because Internet RTT changes with route, queueing, and host scheduling. TCP measures eligible round trips and maintains a smoothed RTT (`SRTT`) and RTT variation (`RTTVAR`). [RFC 6298](https://datatracker.ietf.org/doc/html/rfc6298) uses the initialization

$$
SRTT\leftarrow R,\qquad RTTVAR\leftarrow \frac{R}{2},
$$

and, for later sample $R'$, the updates

$$
RTTVAR\leftarrow (1-\beta)RTTVAR+\beta|SRTT-R'|,
$$

$$
SRTT\leftarrow (1-\alpha)SRTT+\alpha R',
$$

with $\alpha=1/8$ and $\beta=1/4$. Before the standards minimum is applied, the estimate is

$$
RTO=SRTT+\max(G,4\times RTTVAR),
$$

where $G$ is clock granularity. `SRTT` follows the center of recent observations, while the variation term creates a safety margin.

![The raw retransmission timeout remains above the smoothed RTT because it includes an RTT-variation margin.](assets/tcp-rto-estimator.svg){fig-alt="Chart of RTT samples, smoothed RTT, and retransmission timeout with RFC 6298 equations" width="94%"}

#### **Karn's Algorithm and Exponential Backoff**

After a segment is retransmitted, an arriving ACK is ambiguous: it may acknowledge the original or the retransmission. **Karn's algorithm** avoids taking an RTT sample from such an ACK unless timestamps remove the ambiguity. Otherwise one late packet can create a falsely short measurement.

When the retransmission timer expires, TCP retransmits and doubles the RTO. Exponential backoff reduces repeated load when the path is congested or unavailable. A later valid RTT measurement can bring the estimator back toward normal conditions.

#### **Duplicate ACKs and Fast Retransmit**

A timeout is not the only evidence of loss. If later segments arrive while one gap remains, the receiver repeats the same cumulative ACK. In classic TCP loss recovery, three duplicate ACKs trigger **fast retransmit** of the apparently missing segment before the RTO expires. SACK blocks make the evidence more precise by identifying which later byte ranges arrived.

This inference depends on receiver ACK behavior and can be confused by reordering. Modern stacks may use timestamps, SACK, recent-ACK logic, and richer loss detection, but the core reasoning remains: repeated progress beyond one gap is evidence that the gap is more likely loss than ordinary delay.

#### **Spurious Retransmission and Reordering**

A packet can take a slower path and arrive after its replacement. This **spurious retransmission** wastes bandwidth and can interact with congestion control. The receiver must still suppress duplicate bytes. The sender should use conservative timing and reordering-aware evidence instead of interpreting every gap as immediate loss.

| Signal | What the sender can infer | Main uncertainty |
|---|---|---|
| RTO expires | Progress was absent for unusually long | Original data or ACK may only be delayed |
| Repeated cumulative ACK | Later data reached the receiver beyond one gap | Reordering can mimic loss |
| SACK block | Specific later range arrived | Missing range may still be in flight |
| ICMP/path error | A network device reported a problem | Reports can be filtered, delayed, or forged |


In [7]:
class RtoEstimator:
    """RFC 6298 core estimator, exposing both raw and one-second-floor RTO."""

    ALPHA = 1 / 8
    BETA = 1 / 4
    K = 4

    def __init__(self, clock_granularity=0.001):
        self.g = clock_granularity
        self.srtt = None
        self.rttvar = None
        self.raw_rto = 1.0
        self.rto = 1.0

    def observe(self, sample_rtt: float):
        if self.srtt is None:
            self.srtt = sample_rtt
            self.rttvar = sample_rtt / 2
        else:
            old_srtt = self.srtt
            self.rttvar = (
                (1 - self.BETA) * self.rttvar
                + self.BETA * abs(old_srtt - sample_rtt)
            )
            self.srtt = (1 - self.ALPHA) * old_srtt + self.ALPHA * sample_rtt
        self.raw_rto = self.srtt + max(self.g, self.K * self.rttvar)
        self.rto = max(1.0, self.raw_rto)
        return self.srtt, self.rttvar, self.raw_rto, self.rto

    def on_timeout(self):
        # Back off the timer that is actually armed, after applying its floor.
        self.rto *= 2
        return self.rto


estimator = RtoEstimator()
samples_ms = [100, 130, 90, 160, 110, 140]
print("sample  SRTT   RTTVAR  raw RTO  RFC floor RTO")
for sample_ms in samples_ms:
    srtt, variation, raw_rto, effective_rto = estimator.observe(sample_ms / 1000)
    print(
        f"{sample_ms:4}ms  {srtt * 1000:6.1f}  {variation * 1000:7.1f}  "
        f"{raw_rto * 1000:7.1f}ms  {effective_rto:5.3f}s"
    )

print(f"after timeout, backed-off RTO: {estimator.on_timeout():.3f}s")


sample  SRTT   RTTVAR  raw RTO  RFC floor RTO
 100ms   100.0     50.0    300.0ms  1.000s
 130ms   103.8     45.0    283.8ms  1.000s
  90ms   102.0     37.2    250.8ms  1.000s
 160ms   109.3     42.4    278.8ms  1.000s
 110ms   109.4     32.0    237.2ms  1.000s
 140ms   113.2     31.6    239.7ms  1.000s
after timeout, backed-off RTO: 2.000s


### **Flow Control**

#### **Receive Windows and Buffer Occupancy**

Reliable arrival at the host does not mean the application has read the data. TCP stores accepted bytes in a receive buffer. If the network delivers faster than the application consumes, unread bytes accumulate. **Flow control** prevents the sender from overwriting that finite buffer.

A conceptual advertised receive window is

$$
rwnd=\text{receive buffer capacity}-\text{bytes received but not read}.
$$

The receiver includes this window in ACKs. As the application calls `read()`, free space grows and a later window update permits more transmission. The original TCP window field is 16 bits; the window-scale option negotiated during the handshake supports larger buffers on high-BDP paths, as specified by [RFC 7323](https://datatracker.ietf.org/doc/html/rfc7323).

![The receiver advertises its free socket-buffer space, and the sender limits outstanding bytes accordingly.](assets/tcp-flow-control.svg){fig-alt="TCP sender, receiver buffer with ten KiB unread and six KiB free, application drain, and receive-window acknowledgment" width="96%"}

#### **Zero Windows and Persist Timers**

When the buffer is full, the receiver can advertise `rwnd=0`. The sender pauses normal new data. A deadlock is possible if the receiver later advertises a larger window but that update is lost: the receiver waits for data while the sender waits for a window update.

The **persist timer** solves this by causing occasional window probes. A probe elicits a current ACK and window advertisement without resuming unrestricted sending. This is different from an RTO: no unacknowledged data necessarily needs loss recovery.

Tiny repeated window updates can cause many small segments, a behavior related to the **silly window syndrome**. Receiver-side window-update policy and sender-side coalescing can avoid inefficient chatter. Those optimizations must not replace application framing: TCP still exposes a stream, not records.

#### **Flow Control vs Congestion Control**

Flow control protects the **receiver endpoint**. Congestion control protects the **network path and competing traffic**. A sender must obey both. A simplified transmission bound is

$$
\text{bytes in flight}\le \min(rwnd,cwnd),
$$

where `cwnd` is the congestion window. A receiver with abundant memory can advertise a large `rwnd` while a congested path requires a small `cwnd`; a fast path can permit a large `cwnd` while a stalled application forces `rwnd` to zero. Chapter 6 develops how `cwnd` is chosen.


In [8]:
from dataclasses import dataclass


@dataclass
class ReceiveBuffer:
    capacity_kib: int
    unread_kib: int = 0

    @property
    def advertised_window(self) -> int:
        return self.capacity_kib - self.unread_kib

    def receive(self, amount_kib: int) -> int:
        accepted = min(amount_kib, self.advertised_window)
        self.unread_kib += accepted
        return accepted

    def application_read(self, amount_kib: int) -> int:
        consumed = min(amount_kib, self.unread_kib)
        self.unread_kib -= consumed
        return consumed


buffer = ReceiveBuffer(capacity_kib=16)
actions = [
    ("network", 8),
    ("network", 6),
    ("network", 5),  # Only the remaining 2 KiB can be accepted.
    ("application", 10),
    ("network", 6),
]

print("action              accepted/read   unread   advertised rwnd")
for actor, amount in actions:
    if actor == "network":
        changed = buffer.receive(amount)
        label = f"receive {amount:2} KiB"
    else:
        changed = buffer.application_read(amount)
        label = f"read    {amount:2} KiB"
    print(
        f"{label:19} {changed:3} KiB       {buffer.unread_kib:3} KiB"
        f"       {buffer.advertised_window:3} KiB"
    )

congestion_window_kib = 9
print("effective sender window:", min(buffer.advertised_window, congestion_window_kib), "KiB")


action              accepted/read   unread   advertised rwnd
receive  8 KiB        8 KiB         8 KiB         8 KiB
receive  6 KiB        6 KiB        14 KiB         2 KiB
receive  5 KiB        2 KiB        16 KiB         0 KiB
read    10 KiB       10 KiB         6 KiB        10 KiB
receive  6 KiB        6 KiB        12 KiB         4 KiB
effective sender window: 4 KiB


### **QUIC as a Modern Transport**

#### **Transport over UDP and User-Space Evolution**

QUIC, standardized by [RFC 9000](https://datatracker.ietf.org/doc/html/rfc9000), is a secure, multiplexed transport carried in UDP datagrams. "Over UDP" does not mean that QUIC inherits only UDP's behavior. QUIC implements connection establishment, encryption, reliable stream delivery, acknowledgments, loss detection, flow control, and congestion control above the UDP socket interface.

This placement matters operationally. Updating TCP behavior often depends on operating-system kernels and middleboxes. A QUIC implementation can evolve in application or library user space while remaining encapsulated in widely deployable UDP. It still must behave responsibly on the Internet; [RFC 9002](https://datatracker.ietf.org/doc/html/rfc9002) specifies loss detection and congestion-control requirements.

QUIC packet numbers are not reused within a packet-number space. ACK frames can report ranges, and a Probe Timeout (PTO) elicits progress when ACKs or data appear lost. Encryption protects most transport metadata from in-path modification, while exposing only the information needed for routing the UDP packet and demultiplexing the QUIC connection.

#### **Streams, Connection IDs, and Migration**

One QUIC connection can carry many bidirectional or unidirectional **streams**. Bytes are ordered within each stream, but one stream's delivery does not require every earlier byte in a different stream. Stream-level flow control and connection-level flow control jointly protect receiver memory.

QUIC uses connection IDs so a connection is not identified only by an IP address and port tuple. After a mobile device changes from Wi-Fi to cellular or a NAT mapping changes, QUIC can validate the new path and continue the logical connection. Migration is controlled, not automatic trust in any packet claiming an existing ID; path challenge and response frames help validate reachability.

#### **Handshake Integration and Head-of-Line Blocking**

QUIC integrates TLS 1.3 into transport establishment. A first connection can negotiate security and transport parameters together. A resumed connection may use 0-RTT application data, but 0-RTT data can be replayed and therefore must be limited to replay-safe operations.

![Independent QUIC streams can make progress over one connection without requiring cross-stream in-order delivery.](assets/quic-streams.png){fig-alt="Two differently colored independent QUIC streams connect the same pair of computers" width="92%"}

*Figure source: [Daniel Stenberg, HTTP/3 Explained](https://http3-explained.haxx.se/en/the-protocol/feature-inorder), licensed under CC BY 4.0.*

TCP gives HTTP/2 one ordered byte stream. If a TCP segment containing bytes for one HTTP/2 stream is lost, the TCP application boundary cannot expose later bytes for any stream until the gap closes. QUIC places stream offsets in frames. Loss still blocks the missing region of the affected stream, but complete data from another stream can be delivered independently. This removes **cross-stream transport head-of-line blocking**; it does not remove packet loss, application dependencies, or ordering within one stream.

| Dimension | TCP + TLS + HTTP/2 | QUIC + HTTP/3 |
|---|---|---|
| Kernel-visible transport | TCP | UDP substrate; QUIC commonly in user space |
| Security handshake | TLS layered after or combined with TCP setup | TLS 1.3 integrated into QUIC handshake |
| Application multiplexing | HTTP/2 streams inside one TCP byte stream | QUIC transport streams |
| Loss ordering scope | One TCP connection-wide byte stream | Primarily the affected QUIC stream |
| Connection identity | IP/port endpoint tuple | Connection IDs support controlled migration |
| Network friendliness | TCP congestion control | QUIC must implement congestion control too |


In [9]:
class IndependentStream:
    """Reassemble one ordered stream without coupling it to other streams."""

    def __init__(self):
        self.next_offset = 0
        self.fragments = {}
        self.delivered = bytearray()

    def receive(self, offset: int, data: bytes):
        self.fragments[offset] = data
        while self.next_offset in self.fragments:
            fragment = self.fragments.pop(self.next_offset)
            self.delivered.extend(fragment)
            self.next_offset += len(fragment)
        return bytes(self.delivered)


streams = {"A": IndependentStream(), "B": IndependentStream()}

# Stream A is missing its first four bytes, while stream B arrives complete.
print("A after offset 4:", streams["A"].receive(4, b"-page"))
print("B after offset 0:", streams["B"].receive(0, b"style.css"))
print("A after repair:  ", streams["A"].receive(0, b"home"))


A after offset 4: b''
B after offset 0: b'style.css'
A after repair:   b'home-page'


### **Implementing TCP and UDP Socket Programs**

Socket APIs expose transport behavior, but they do not remove it. A robust program checks return values, bounds memory, handles timeouts, closes resources, validates peer data, and defines an application protocol.

For UDP, a minimal request path is:

```text
create datagram socket
optionally bind local address and port
encode one complete application message
sendto(message, peer)
recvfrom(maximum_size) -> message and source endpoint
validate source, size, identifier, and contents
```

The receiver must decide how to handle duplicates, reordering, loss, retry amplification, and messages larger than the path can safely carry. Setting a socket timeout does not make the operation reliable; it only returns control to the application.

For TCP, a server broadly performs `socket() -> bind() -> listen() -> accept()`, while a client performs `socket() -> connect()`. Both then exchange stream bytes. Production code must treat `send()` as potentially partial and `recv()` as returning any positive number of currently available bytes. A length-prefixed record is commonly encoded as

$$
\text{frame}=\text{fixed-width length}\ ||\ \text{payload}.
$$

The following examples use only local sockets, so executing this notebook sends nothing onto the external network.


In [10]:
import socket


# Bind to port 0 so the operating system selects an unused local UDP port.
with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as receiver, \
     socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as sender:
    receiver.bind(("127.0.0.1", 0))
    receiver.settimeout(1.0)
    local_endpoint = receiver.getsockname()

    request = b"id=7;question=transport"
    sender.sendto(request, local_endpoint)
    response, source = receiver.recvfrom(2048)

print("received:", response)
print("source is loopback:", source[0] == "127.0.0.1")
print("one sendto produced one received datagram:", response == request)


received: b'id=7;question=transport'
source is loopback: True
one sendto produced one received datagram: True


In [11]:
import socket
import struct


def encode_frame(payload: bytes) -> bytes:
    return struct.pack("!I", len(payload)) + payload


def decode_frames(buffer: bytearray):
    messages = []
    while len(buffer) >= 4:
        length = struct.unpack("!I", buffer[:4])[0]
        if len(buffer) < 4 + length:
            break
        messages.append(bytes(buffer[4:4 + length]))
        del buffer[:4 + length]
    return messages


left, right = socket.socketpair()
with left, right:
    # Two application messages are written together; TCP would expose only bytes.
    left.sendall(encode_frame(b"alpha") + encode_frame(b"beta-beta"))
    left.shutdown(socket.SHUT_WR)

    receive_buffer = bytearray()
    decoded = []
    while True:
        chunk = right.recv(5)  # Deliberately unrelated to frame boundaries.
        if not chunk:
            break
        receive_buffer.extend(chunk)
        decoded.extend(decode_frames(receive_buffer))

print("decoded messages:", decoded)
print("unconsumed bytes:", len(receive_buffer))


decoded messages: [b'alpha', b'beta-beta']
unconsumed bytes: 0


### **Building a Minimal Reliable Byte Stream**

A teaching transport can now combine the chapter's mechanisms. It is not a replacement for TCP or QUIC, because production protocols must handle wraparound, security, path MTU, congestion, adversarial input, many simultaneous losses, and decades of edge cases. Its value is to make the dependency chain explicit.

```text
split application bytes into segments with byte offsets
attach checksum and retain each unacknowledged segment
permit only segments inside the current send window
receiver verifies, buffers by offset, and delivers only contiguous bytes
receiver returns cumulative ACK plus received ranges
sender removes acknowledged data and selectively retransmits gaps
bound bytes in flight by receiver credit and congestion permission
```

The example below sends `reliable-stream` in four-byte segments. The channel drops the segment at byte offset 4 on its first attempt and delivers later segments out of order. The receiver's cumulative ACK stops at 4 while later ranges remain buffered. Retransmitting only offset 4 closes the gap, releases all buffered bytes, and advances the ACK to the stream length.


In [12]:
from dataclasses import dataclass
import zlib


@dataclass(frozen=True)
class Segment:
    offset: int
    payload: bytes
    checksum: int

    @classmethod
    def create(cls, offset: int, payload: bytes):
        protected = offset.to_bytes(4, "big") + payload
        return cls(offset, payload, zlib.crc32(protected))

    def valid(self) -> bool:
        protected = self.offset.to_bytes(4, "big") + self.payload
        return zlib.crc32(protected) == self.checksum


class MinimalByteReceiver:
    def __init__(self):
        self.next_offset = 0
        self.fragments = {}
        self.application = bytearray()

    def receive(self, segment: Segment):
        if segment.valid() and segment.offset >= self.next_offset:
            self.fragments.setdefault(segment.offset, segment.payload)
        while self.next_offset in self.fragments:
            payload = self.fragments.pop(self.next_offset)
            self.application.extend(payload)
            self.next_offset += len(payload)
        sacks = sorted(
            (offset, offset + len(payload))
            for offset, payload in self.fragments.items()
        )
        return self.next_offset, sacks


data = b"reliable-stream"
segments = [Segment.create(offset, data[offset:offset + 4]) for offset in range(0, len(data), 4)]
receiver = MinimalByteReceiver()

# First pass: offset 4 is lost; later segments are buffered out of order.
for segment in [segments[0], segments[2], segments[3]]:
    ack, sacks = receiver.receive(segment)
    print(f"arrival offset={segment.offset:2} -> ACK={ack:2}, SACK={sacks}")

# Selective repair of the first missing range releases the buffered suffix.
ack, sacks = receiver.receive(segments[1])
print(f"repair  offset={segments[1].offset:2} -> ACK={ack:2}, SACK={sacks}")
print("application received:", bytes(receiver.application))


arrival offset= 0 -> ACK= 4, SACK=[]
arrival offset= 8 -> ACK= 4, SACK=[(8, 12)]
arrival offset=12 -> ACK= 4, SACK=[(8, 12), (12, 15)]
repair  offset= 4 -> ACK=15, SACK=[]
application received: b'reliable-stream'


### **Observing and Troubleshooting Transport Behavior**

A transport failure is easiest to diagnose by separating endpoint state, packets, and application expectations.

On Windows, useful commands include:

```powershell
Get-NetTCPConnection
Get-NetUDPEndpoint
netstat -ano
Test-NetConnection example.com -Port 443
```

In Wireshark, start with one flow and then inspect sequence analysis:

```text
tcp.stream eq 3
tcp.analysis.retransmission
tcp.analysis.duplicate_ack
tcp.window_size_value == 0
udp.port == 53
quic
```

| Symptom | Evidence to inspect | Likely classes of cause |
|---|---|---|
| TCP connection never establishes | SYN, SYN-ACK, RST, retransmission timing | service absent, firewall, asymmetric path, backlog pressure |
| Transfer pauses with zero window | advertised window and application reads | slow or stalled receiving process |
| Repeated retransmissions | sequence gaps, SACK blocks, RTT, capture location | path loss, reordering, MTU issue, receiver overload |
| UDP request times out | request ID, destination, reply source, ICMP | loss, wrong endpoint, server delay, application retry policy |
| HTTP/3 falls back | UDP reachability, QUIC handshake, TLS alerts | UDP blocking, version negotiation, certificate or policy failure |
| Data appears merged or split | application framing and receive loop | incorrect assumption that TCP preserves messages |

A capture point matters. NIC offload can make host captures show segments larger than the packets on the wire, and capturing only one side cannot prove what the other endpoint received. Correlate packet traces with socket state, application logs, and timing.

### **Summary**

- Ports and sockets extend IP host delivery to specific processes and simultaneous flows.
- UDP preserves messages and supplies a checksum, but reliability, ordering, flow control, and congestion response must come from above.
- Sequence numbers, ACKs, timers, retransmission, and duplicate suppression form the minimal reliable-transfer toolkit.
- Sliding windows fill a path's bandwidth-delay product; Selective Repeat avoids the broad replay used by Go-Back-N.
- TCP turns packets into an ordered byte stream with cumulative ACKs, optional SACK, a connection state machine, adaptive RTO, and receiver flow control.
- QUIC rebuilds secure reliable transport over UDP, uses independent streams and connection IDs, and avoids cross-stream transport head-of-line blocking.
- Flow control answers whether the receiver can accept more. Chapter 6 asks whether the network path should accept more and develops congestion control, queueing, fairness, and resource sharing.
